# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [ ]:
import os
import warnings
import tiktoken

from dotenv import load_dotenv
from IPython.display import Markdown, display

from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain_classic.agents import AgentExecutor, create_react_agent
from langchainhub import Client


load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.invoke({"query": query})

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:2500])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]


agent = create_react_agent(llm, tools, prompt)
# react agent before making a decision, it will take time to do the reacting part 


# Create agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations = 10 
)



### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
# Your prompt here

prompt_1 = """
In a structured response in a structured Markdown report, investigate and summarize the latest advancements in generative AI applied to healthcare in 2025. 
- Main heading: latest advancements in generative AI 
- Focus on the application of generative AI to healthcare
- Only in the year of 2025
Keep it concise.
"""
# Run it
response_1 = agent_executor.invoke({"input": prompt_1})


In [ ]:
print(response_1["output"])


### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [ ]:
# Your prompt here
prompt_2 = """
Provide a SHORT well-structured Markdown report tracking 2025's top emerging AI startups and their innovations.
Use headings for each company, bullet points for product highlights, and include relevant sources URLs in Markdown link format. 
Format it for easy reading and professional presentation.
- Maximum 5 companies
- Just 3 bullet points per company
- Very concise
- Focus ONLY on major innovation

"""

# Run it
response_2 = agent_executor.invoke({"input": prompt_2})


In [ ]:
display(Markdown(response_2["output"]))


### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [ ]:
# Your prompt here
prompt_3 = """
Provide a detailed, well-structured Markdown report comparing the key features, pricing, and reviews of OpenAI's ChatGPA Team and Anthropic's Claude Pro. 
Use headings for each tech product, bullet points for key features, pricing and reviews. 
Format it for easy reading and professional presentation.
- Be very concise
- Maximum 5 key features
"""

# Run it
response_3 = agent_executor.invoke({"input": prompt_3})

In [ ]:
display(Markdown(response_3["output"]))



## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
